# Week 5 L1 — Introduction to PyMongo — solution

In the VS Code playground you wrote queries like this:

```javascript
db.restaurants.find({ borough: "Brooklyn" }).limit(5)
```

Today you work with MongoDB **from Python**, using **PyMongo**.

- **Section 1:** connect, load data, read documents, and create, update and delete them (CRUD).
- **Section 2:** ask better questions with operators, several conditions, regex, and the smallest and largest values.

Each **Part A** introduces ideas with examples; each **Part B** is for you to do.

## Set up

### Install the packages

In [1]:
%pip install pymongo python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Settings

Your connection string and database name are kept in a file called `.env`, in the same folder as this notebook, so they are not written into the notebook itself:

```
MONGO_URI=paste the connection string from Canvas here
DB_NAME=MOD008909_your_student_id
```

In [2]:
# ---- CONFIG ----
import os
from dotenv import load_dotenv

load_dotenv()   # reads the .env file in the notebook's folder

MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("DB_NAME")

if not MONGO_URI or not DB_NAME:
    print("MONGO_URI or DB_NAME not found: check that .env is in the same folder as this notebook")
else:
    print("Settings loaded for database:", DB_NAME)

Settings loaded for database: SK2063


### Your files

The CSV files sit in the same folder as this notebook.

In [3]:
folder_path = "./"

print(os.listdir(folder_path))

['.env', '.venv', 'area_population_clean.csv', 'community_health_activity_clean.csv', 'community_health_activity_with_population.csv', 'MOD008909_Descriptive_Statistics_and_Data_Mining_Tasks.pptx', 'README.md', 'sexDensitiesFemale.csv', 'sexDensitiesMale.csv', 'Week5_L1_Canvas.ipynb', 'Week5_L1_Introduction_to_PyMongo.ipynb', 'Week5_L1_Introduction_to_PyMongo_Solution.ipynb', 'Week5_L2_Data_Mining_Descriptive_Analysis.ipynb', 'Week5_L2_Data_Mining_Descriptive_Analysis_Solution.ipynb']


---
# Section 1 — PyMongo basics and CRUD

# Part A (1) — Concepts

## 1. What PyMongo is

Python cannot talk to MongoDB on its own. **PyMongo** is a *driver*: a library that sends your requests to the database and brings the answers back as Python objects.

The structure is the same as in the playground:

| Level | In the playground | In PyMongo |
|---|---|---|
| the server | (already connected) | `client = MongoClient(MONGO_URI)` |
| a database | `use MOD008909_...` | `db = client["MOD008909_..."]` |
| a collection | `db.sexDensitiesMale` | `db["sexDensitiesMale"]` |
| a document | `{ CODE: "E06000031", ... }` | `{"CODE": "E06000031", ...}` (a Python dictionary) |

In [4]:
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

print("Databases:", client.list_database_names())

Databases: ['SK2063', 'assignment', 'bigdata_lab', 'gdhi_life_mart', 'mongodbVSCodePlaygroundDB', 'sample_mflix', 'sample_restaurants', 'student_records', 'admin', 'local']


### Load the two files into MongoDB

This cell reads each CSV into pandas, turns every row into a dictionary with `to_dict("records")`, and stores them all in their own collection in one go with `insert_many`. For more, see the [PyMongo tutorial](https://pymongo.readthedocs.io/en/stable/tutorial.html).

In [5]:
import pandas as pd
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
db = client[DB_NAME]


def load_csv_to_collection(file_name, collection_name):
    df = pd.read_csv(folder_path + file_name)        # read the CSV into a table
    collection = db[collection_name]
    collection.drop()                                # empty it first, so re-running does not double the data
    collection.insert_many(df.to_dict("records"))    # one document per row
    print(collection_name, ":", collection.count_documents({}), "documents")
    return collection


male = load_csv_to_collection("sexDensitiesMale.csv", "sexDensitiesMale")
female = load_csv_to_collection("sexDensitiesFemale.csv", "sexDensitiesFemale")

sexDensitiesMale : 439 documents
sexDensitiesFemale : 439 documents


Expected: 439 documents in each.

Each document is one area. `CODE` is the ONS area code, `Location` its name, `AllAges` the total, and the other fields are age bands: `<18`, `19-25`, `26-30`, `31-35`, `35-45`, `45-60`, `>61`.

In [6]:
print("Collections in my database:", db.list_collection_names())

Collections in my database: ['activity', 'places', 'activity_with_population', 'sexDensitiesMale', 'sexDensitiesFemale', 'population']


## 2. Playground → Python: the translation rules

| Playground (JavaScript) | PyMongo (Python) |
|---|---|
| `{ Location: "Cambridge" }` | `{"Location": "Cambridge"}` — field names need quotes |
| `{ AllAges: { $gt: 500000 } }` | `{"AllAges": {"$gt": 500000}}` — operators need quotes too |
| `true`, `false`, `null` | `True`, `False`, `None` |
| `findOne()`, `countDocuments()` | `find_one()`, `count_documents()` |
| `.sort({ AllAges: -1 })` | `.sort("AllAges", -1)` |

## 3. One document: `find_one()`

`find_one()` returns the first document that matches the filter, as a Python dictionary. MongoDB has added an `_id` field.

In [7]:
male.find_one({"Location": "Peterborough"})

{'_id': ObjectId('6ad746b90be16a102882607b'),
 'CODE': 'E06000031',
 'Location': 'Peterborough',
 'AllAges': 94952,
 '<18': 25162,
 '19-25': 8419,
 '26-30': 7748,
 '31-35': 7590,
 '35-45': 13249,
 '45-60': 16944,
 '>61': 15840}

Because the result is a dictionary, you can pick out one field:

In [8]:
doc = male.find_one({"Location": "Peterborough"})
print("Males in Peterborough:", doc["AllAges"])

Males in Peterborough: 94952


## 4. Many documents: `find()` and the cursor

`find()` does **not** return the documents straight away. It returns a **cursor**: a pointer that hands the documents over one at a time. Printing it shows the cursor, not the data:

In [9]:
print(male.find({"Location": "Peterborough"}))

To see the documents, **loop** over the cursor, or turn it into a list with `list()`.

In [10]:
for doc in male.find({"Location": "Peterborough"}):
    print(doc)

{'_id': ObjectId('6ad746b90be16a102882607b'), 'CODE': 'E06000031', 'Location': 'Peterborough', 'AllAges': 94952, '<18': 25162, '19-25': 8419, '26-30': 7748, '31-35': 7590, '35-45': 13249, '45-60': 16944, '>61': 15840}


## 5. Projection: choose the fields

The second dictionary in `find()` says which fields to show: `1` to include, `0` to leave out. `_id` is shown unless you turn it off.

In [11]:
for doc in female.find({"Location": "Cambridge"}, {"_id": 0, "Location": 1, "AllAges": 1, ">61": 1}):
    print(doc)

{'Location': 'Cambridge', 'AllAges': 61971, '>61': 10677}


## 6. `limit`, `sort` and counting

- `.limit(n)` keeps the first `n` documents.
- `.sort(field, 1)` sorts smallest first; `.sort(field, -1)` largest first.
- `count_documents(filter)` counts the matches. `{}` is an empty filter: every document.

In [12]:
projection = {"_id": 0, "Location": 1, "AllAges": 1}

for doc in male.find({}, projection).sort("AllAges", -1).limit(3):
    print(doc)

{'Location': 'UNITED KINGDOM', 'AllAges': 31793752}
{'Location': 'GREAT BRITAIN', 'AllAges': 30891041}
{'Location': 'ENGLAND AND WALES', 'AllAges': 28294511}


In [13]:
print("Male documents:", male.count_documents({}))
print("Areas with more than 500,000 males:", male.count_documents({"AllAges": {"$gt": 500000}}))

Male documents: 439
Areas with more than 500,000 males: 29


The three largest are the United Kingdom, Great Britain, and England and Wales: the files hold countries, regions and counties as well as local authorities.

## 7. CRUD: the four things you do with data

You have **read** documents. A database supports four basic operations:

| Operation | Meaning | Playground | PyMongo |
|---|---|---|---|
| **C**reate | add new documents | `insertOne()`, `insertMany()` | `insert_one()`, `insert_many()` |
| **R**ead | get documents back | `find()`, `findOne()` | `find()`, `find_one()` |
| **U**pdate | change documents | `updateOne()`, `updateMany()` | `update_one()`, `update_many()` |
| **D**elete | remove documents | `deleteOne()`, `deleteMany()` | `delete_one()`, `delete_many()` |

The examples use a small collection of your own, `places`, so the male and female data stay as they are.

### Create: `insert_one()` and `insert_many()`

A document is a Python dictionary. The collection is created the moment the first document goes in. `insert_one()` returns a result whose `inserted_id` is the `_id` MongoDB added; `insert_many()` takes a **list** and returns `inserted_ids`.

In [14]:
places = db["places"]
places.drop()   # start empty, so this notebook can be run again

result = places.insert_one({"name": "Peterborough", "county": "Cambridgeshire", "population": 215700})
print("Inserted id:", result.inserted_id)

result = places.insert_many([
    {"name": "Cambridge", "county": "Cambridgeshire", "population": 145700},
    {"name": "Norwich", "county": "Norfolk", "population": 144000},
    {"name": "Ely", "county": "Cambridgeshire", "population": 20300},
])
print("Inserted:", len(result.inserted_ids))
print("Documents in places:", places.count_documents({}))

Inserted id: 6ad746bd0be16a102882633d
Inserted: 3
Documents in places: 4


Documents in one collection do not need the same fields. `$exists` finds those that have a field:

In [15]:
places.insert_one({"name": "Wisbech", "county": "Cambridgeshire", "population": 31600, "market_town": True})
print("With a market_town field:", places.count_documents({"market_town": {"$exists": True}}))

for doc in places.find({}, {"_id": 0}):
    print(doc)

With a market_town field: 1
{'name': 'Peterborough', 'county': 'Cambridgeshire', 'population': 215700}
{'name': 'Cambridge', 'county': 'Cambridgeshire', 'population': 145700}
{'name': 'Norwich', 'county': 'Norfolk', 'population': 144000}
{'name': 'Ely', 'county': 'Cambridgeshire', 'population': 20300}
{'name': 'Wisbech', 'county': 'Cambridgeshire', 'population': 31600, 'market_town': True}


## 8. Update: `update_one()` and `update_many()`

An update takes **two** dictionaries: a **filter** (which documents, as in `find()`) and an **update** with an operator.

| Operator | What it does | Example |
|---|---|---|
| `$set` | set a field (adds it if it is not there) | `{"$set": {"population": 216000}}` |
| `$inc` | add to a number | `{"$inc": {"population": 500}}` |
| `$unset` | remove a field | `{"$unset": {"market_town": ""}}` |

`update_one()` changes the first match; `update_many()` changes every match. The result says how many documents **matched** and how many were **modified**.

In [16]:
result = places.update_one({"name": "Ely"}, {"$set": {"population": 20500}})
print("update_one  matched:", result.matched_count, "| modified:", result.modified_count)

result = places.update_many({"county": "Cambridgeshire"}, {"$set": {"region": "East of England"}})
print("update_many matched:", result.matched_count, "| modified:", result.modified_count)

result = places.update_one({"name": "Norwich"}, {"$inc": {"population": 1000}})
print(places.find_one({"name": "Norwich"}, {"_id": 0}))

update_one  matched: 1 | modified: 1
update_many matched: 4 | modified: 4
{'name': 'Norwich', 'county': 'Norfolk', 'population': 145000}


Without an operator, PyMongo refuses the update:

In [17]:
try:
    places.update_one({"name": "Ely"}, {"population": 20600})
except ValueError as error:
    print("Refused:", error)

Refused: update only works with $ operators


## 9. Delete: `delete_one()` and `delete_many()`

Delete takes a filter, the same as `find()`. An **empty** filter `{}` matches every document, so `delete_many({})` empties the collection, with no undo.

A safe routine: **look** with `find()`, **count** with `count_documents()`, then delete with **the same filter**, and check `deleted_count`.

In [18]:
target = {"county": "Norfolk"}

for doc in places.find(target, {"_id": 0, "name": 1}):
    print("Will delete:", doc)
print("Count:", places.count_documents(target))

result = places.delete_many(target)
print("Deleted:", result.deleted_count, "| left:", places.count_documents({}))

Will delete: {'name': 'Norwich'}
Count: 1
Deleted: 1 | left: 4


---
# Part B (1) — Your turn

## Task 1: Translate it

Each line is a playground query. Write it in PyMongo in the cell below it and run it.

**1a.**
```javascript
db.sexDensitiesFemale.findOne({ Location: "Norwich" })
```

In [19]:
female.find_one({"Location": "Norwich"})

{'_id': ObjectId('6ad746bb0be16a1028826259'),
 'CODE': 'E07000148',
 'Location': 'Norwich',
 'AllAges': 69383,
 '<18': 13329,
 '19-25': 11798,
 '26-30': 6264,
 '31-35': 5293,
 '35-45': 8153,
 '45-60': 10707,
 '>61': 13839}

**1b.**
```javascript
db.sexDensitiesMale.find({ Location: "Cardiff" }, { _id: 0, Location: 1, "19-25": 1 })
```

In [20]:
for doc in male.find({"Location": "Cardiff"}, {"_id": 0, "Location": 1, "19-25": 1}):
    print(doc)

{'Location': 'Cardiff', '19-25': 27114}


**1c.**
```javascript
db.sexDensitiesMale.find({}, { _id: 0, Location: 1 }).limit(5)
```

In [21]:
for doc in male.find({}, {"_id": 0, "Location": 1}).limit(5):
    print(doc)

{'Location': 'UNITED KINGDOM'}
{'Location': 'GREAT BRITAIN'}
{'Location': 'ENGLAND AND WALES'}
{'Location': 'ENGLAND'}
{'Location': 'NORTH EAST'}


**1d.**
```javascript
db.sexDensitiesFemale.countDocuments({})
```
Expected: 439

In [22]:
female.count_documents({})

439

**1e.**
```javascript
db.sexDensitiesFemale.find({}, { _id: 0, Location: 1, ">61": 1 }).sort({ ">61": -1 }).limit(3)
```

In [23]:
for doc in female.find({}, {"_id": 0, "Location": 1, ">61": 1}).sort(">61", -1).limit(3):
    print(doc)

{'Location': 'UNITED KINGDOM', '>61': 7694896}
{'Location': 'GREAT BRITAIN', '>61': 7497958}
{'Location': 'ENGLAND AND WALES', '>61': 6828123}


**1f.**
```javascript
db.sexDensitiesMale.countDocuments({ AllAges: { $gt: 200000 } })
```
Expected: 62

In [24]:
male.count_documents({"AllAges": {"$gt": 200000}})

62

## Task 2: Predict, then run

For each one, write your prediction **before** you run it.

**2a.** What does this print?
```python
print(male.find({"Location": "Peterborough"}))
```
*My prediction:*

**2b.** How many documents does this print?
```python
for doc in male.find({}).limit(3):
    print(doc)
```
*My prediction:*

**2c.** Which fields appear?
```python
female.find_one({"Location": "Peterborough"}, {"_id": 0, "Location": 1})
```
*My prediction:*

**2d.** Are there more males or females of all ages in Peterborough? Write the two `find_one` lines that answer it.

In [25]:
m = male.find_one({"Location": "Peterborough"})["AllAges"]
f = female.find_one({"Location": "Peterborough"})["AllAges"]
print("Males:", m, "| Females:", f)

Males: 94952 | Females: 95509


**Answers.** 2a prints a `Cursor` object, not data. 2b prints 3 documents. 2c shows only `Location` (no `_id`). 2d: 94,952 males and 95,509 females: slightly more females.

## Task 3: Fix the bug

Each snippet has **one** mistake. Write the corrected version in the cell below it.

**3a.**
```python
male.find_one({Location: "Cambridge"})
```

In [26]:
# Field names need quotes in Python
male.find_one({"Location": "Cambridge"})

{'_id': ObjectId('6ad746b90be16a102882607f'),
 'CODE': 'E07000008',
 'Location': 'Cambridge',
 'AllAges': 66544,
 '<18': 12088,
 '19-25': 14501,
 '26-30': 7006,
 '31-35': 5925,
 '35-45': 8505,
 '45-60': 9709,
 '>61': 8810}

**3b.**
```python
male.count_documents({"AllAges": $gt: 100000})
```

In [27]:
# The operator and its value go inside their own dictionary, with quotes
male.count_documents({"AllAges": {"$gt": 100000}})

144

**3c.**
```python
for doc in male.find({}).sort({"AllAges": -1}).limit(3):
    print(doc)
```

In [28]:
# In PyMongo, sort takes the field and the direction, not a dictionary
for doc in male.find({}).sort("AllAges", -1).limit(3):
    print(doc)

{'_id': ObjectId('6ad746b90be16a1028825fcf'), 'CODE': 'K02000001', 'Location': 'UNITED KINGDOM', 'AllAges': 31793752, '<18': 7412279, '19-25': 3041541, '26-30': 2184056, '31-35': 2153921, '35-45': 4195894, '45-60': 6299308, '>61': 6506753}
{'_id': ObjectId('6ad746b90be16a1028825fd0'), 'CODE': 'K03000001', 'Location': 'GREAT BRITAIN', 'AllAges': 30891041, '<18': 7177668, '19-25': 2955183, '26-30': 2122744, '31-35': 2094404, '35-45': 4077333, '45-60': 6120583, '>61': 6343126}
{'_id': ObjectId('6ad746b90be16a1028825fd1'), 'CODE': 'K04000001', 'Location': 'ENGLAND AND WALES', 'AllAges': 28294511, '<18': 6617392, '19-25': 2701849, '26-30': 1948958, '31-35': 1927275, '35-45': 3741714, '45-60': 5563394, '>61': 5793929}


**3d.**
```python
male.findOne({"Location": "Leeds"})
```

In [29]:
# Python spelling: find_one, not findOne
male.find_one({"Location": "Leeds"})

{'_id': ObjectId('6ad746b90be16a1028826024'),
 'CODE': 'E08000035',
 'Location': 'Leeds',
 'AllAges': 376004,
 '<18': 87252,
 '19-25': 50265,
 '26-30': 29091,
 '31-35': 27168,
 '35-45': 48902,
 '45-60': 67196,
 '>61': 66130}

**3e.** This returns `None`. Why?
```python
male.find_one({"location": "Leeds"})
```

In [30]:
# Field names are case-sensitive: the field is "Location", not "location"
male.find_one({"Location": "Leeds"})

{'_id': ObjectId('6ad746b90be16a1028826024'),
 'CODE': 'E08000035',
 'Location': 'Leeds',
 'AllAges': 376004,
 '<18': 87252,
 '19-25': 50265,
 '26-30': 29091,
 '31-35': 27168,
 '35-45': 48902,
 '45-60': 67196,
 '>61': 66130}

## Task 4: CRUD on your own collection

**4a.** Create a collection `my_places` (drop it first). Insert **one** place you know with `name`, `county` and `population`, then **three** more in one call.
**4b.** Update one place's population with `$set`, and add `"visited": True` to every place in one county with `update_many`. Print matched and modified each time.
**4c.** Delete one place safely: look, count, delete with the same filter, check.
**4d.** Print what is left, without `_id`.

Expected: 4 documents after 4a, 3 after 4c.

In [31]:
my_places = db["my_places"]
my_places.drop()

my_places.insert_one({"name": "Peterborough", "county": "Cambridgeshire", "population": 215700})
my_places.insert_many([
    {"name": "King's Lynn", "county": "Norfolk", "population": 42800},
    {"name": "Stamford", "county": "Lincolnshire", "population": 20600},
    {"name": "Huntingdon", "county": "Cambridgeshire", "population": 25000},
])
print("4a documents:", my_places.count_documents({}))

r = my_places.update_one({"name": "Stamford"}, {"$set": {"population": 21000}})
print("4b update_one matched:", r.matched_count, "| modified:", r.modified_count)
r = my_places.update_many({"county": "Cambridgeshire"}, {"$set": {"visited": True}})
print("4b update_many matched:", r.matched_count, "| modified:", r.modified_count)

target = {"name": "King's Lynn"}
print("4c would delete:", my_places.count_documents(target))
r = my_places.delete_one(target)
print("4c deleted:", r.deleted_count, "| left:", my_places.count_documents({}))

for doc in my_places.find({}, {"_id": 0}):
    print("4d", doc)

4a documents: 4
4b update_one matched: 1 | modified: 1
4b update_many matched: 2 | modified: 2
4c would delete: 1
4c deleted: 1 | left: 3
4d {'name': 'Peterborough', 'county': 'Cambridgeshire', 'population': 215700, 'visited': True}
4d {'name': 'Stamford', 'county': 'Lincolnshire', 'population': 21000}
4d {'name': 'Huntingdon', 'county': 'Cambridgeshire', 'population': 25000, 'visited': True}


---
# Section 2 — Asking better questions: filters and regex

# Part A (2) — Concepts

## 10. Comparison operators

An operator goes inside its own dictionary: `{field: {operator: value}}`.

| Operator | Meaning |
|---|---|
| `$gt`, `$gte` | greater than, greater than or equal |
| `$lt`, `$lte` | less than, less than or equal |
| `$ne` | not equal |
| `$in` | equal to any value in a list |

In [32]:
print("More than 200,000 males:  ", male.count_documents({"AllAges": {"$gt": 200000}}))
print("Fewer than 5,000 males <18:", male.count_documents({"<18": {"$lt": 5000}}))

projection = {"_id": 0, "Location": 1, "AllAges": 1}
for doc in male.find({"Location": {"$in": ["Peterborough", "Cambridge", "Norwich"]}}, projection):
    print(doc)

More than 200,000 males:   62
Fewer than 5,000 males <18: 9
{'Location': 'Peterborough', 'AllAges': 94952}
{'Location': 'Cambridge', 'AllAges': 66544}
{'Location': 'Norwich', 'AllAges': 68089}


## 11. Several conditions

Put several fields in **one** filter dictionary and **all** must be true (AND).

`$or` takes a **list** of filters; a document matches if **any** of them is true.

In [33]:
# AND: Welsh areas (codes starting W06) with more than 50,000 males
both = {"CODE": {"$regex": "^W06"}, "AllAges": {"$gt": 50000}}
print("Welsh areas with > 50,000 males:", male.count_documents(both))

# OR: Peterborough or Cambridge
either = {"$or": [{"Location": "Peterborough"}, {"Location": "Cambridge"}]}
for doc in male.find(either, projection):
    print(doc)

Welsh areas with > 50,000 males: 15
{'Location': 'Peterborough', 'AllAges': 94952}
{'Location': 'Cambridge', 'AllAges': 66544}


## 12. Text patterns with regex

You have met regular expressions already. In a filter, `$regex` matches text by pattern:

- `"^W06"`: starts with W06 (all Welsh unitary authorities)
- `"shire"`: contains "shire" anywhere
- `"$options": "i"`: ignore upper and lower case

Python's own `re.compile()` works too.

In [34]:
import re

print("Codes starting E09 (London boroughs):", male.count_documents({"CODE": {"$regex": "^E09"}}))
print("'shire' (case-sensitive):  ", male.count_documents({"Location": {"$regex": "shire"}}))
print("'shire' (any case):        ", male.count_documents({"Location": {"$regex": "shire", "$options": "i"}}))
print("Same with re.compile:      ", male.count_documents({"Location": re.compile("shire", re.IGNORECASE)}))

Codes starting E09 (London boroughs): 33
'shire' (case-sensitive):   65
'shire' (any case):         66
Same with re.compile:       66


The two counts differ by one: `YORKSHIRE AND THE HUMBER` is written in capitals.

## 13. Smallest and largest with `sort` + `limit`

Sort smallest-first and keep one document: that is the **minimum**. Sort largest-first: the **maximum**.

Minimum and maximum are two of the measures used to **summarise** a dataset, and summarising is the first **data mining** task. Together they show the full spread of the values: the range.

In [35]:
smallest = male.find({}, projection).sort("AllAges", 1).limit(1)
largest = male.find({}, projection).sort("AllAges", -1).limit(1)

print("Minimum:", list(smallest))
print("Maximum:", list(largest))

Minimum: [{'Location': 'Isles of Scilly', 'AllAges': 1148}]
Maximum: [{'Location': 'UNITED KINGDOM', 'AllAges': 31793752}]


The maximum is the whole United Kingdom, because the file mixes countries and regions with local areas. To compare **local authorities only**, filter by their code prefixes first:

In [36]:
local_authorities = {"CODE": {"$regex": "^(E06|E07|E08|E09|W06|S12|N09)"}}
print("Local authorities:", male.count_documents(local_authorities))

largest_la = male.find(local_authorities, projection).sort("AllAges", -1).limit(1)
print("Largest local authority:", list(largest_la))

Local authorities: 390
Largest local authority: [{'Location': 'Birmingham', 'AllAges': 543580}]


## 14. From MongoDB to pandas

Filter in the database, then bring the result into pandas for anything else: `pd.DataFrame(list(...))`.

In [37]:
welsh = pd.DataFrame(list(male.find({"CODE": {"$regex": "^W06"}}, {"_id": 0, "Location": 1, "AllAges": 1})))
welsh.sort_values("AllAges", ascending=False).head()

,Location,AllAges
14,Cardiff,174278
10,Swansea,120029
15,Rhondda Cynon Taf,116076
9,Carmarthenshire,90484
17,Caerphilly,88097


---
# Part B (2) — Your turn

## Task 5: The query ladder

Each question is a little harder than the last. Use `count_documents` for the counts; the expected answer is given so you can check yourself.

| # | Question | Expected |
|---|---|---|
| 1 | Male documents with `AllAges` greater than 200,000 | 62 |
| 2 | Male documents with fewer than 5,000 under-18s (`<18`) | 9 |
| 3 | Female documents with `AllAges` of 100,000 or more | 150 |
| 4 | Male documents for Peterborough, Cambridge or Norwich (use `$in`) | 3 |
| 5 | London boroughs: male codes starting `E09` | 33 |
| 6 | Welsh areas (`W06`) with more than 50,000 males | 15 |
| 7 | Areas that are London boroughs **or** Welsh (`$or`) | 55 |
| 8 | Male locations whose name starts with `North` | 18 |
| 9 | Male locations containing `shire`, any case | 66 |
| 10 | Male documents where `19-25` is greater than `26-30` (hint: this needs both fields, so think about whether a filter can do it; if not, use pandas) | 418 |

In [38]:
print(1, male.count_documents({"AllAges": {"$gt": 200000}}))
print(2, male.count_documents({"<18": {"$lt": 5000}}))
print(3, female.count_documents({"AllAges": {"$gte": 100000}}))
print(4, male.count_documents({"Location": {"$in": ["Peterborough", "Cambridge", "Norwich"]}}))
print(5, male.count_documents({"CODE": {"$regex": "^E09"}}))
print(6, male.count_documents({"CODE": {"$regex": "^W06"}, "AllAges": {"$gt": 50000}}))
print(7, male.count_documents({"$or": [{"CODE": {"$regex": "^E09"}}, {"CODE": {"$regex": "^W06"}}]}))
print(8, male.count_documents({"Location": {"$regex": "^North"}}))
print(9, male.count_documents({"Location": {"$regex": "shire", "$options": "i"}}))

# 10 compares two fields in the same document: a simple filter compares a field with a value,
# so bring the data into pandas and compare there.
all_male = pd.DataFrame(list(male.find({}, {"_id": 0})))
print(10, (all_male["19-25"] > all_male["26-30"]).sum())

1 62
2 9
3 150
4 3
5 33
6 15
7 55
8 18
9 66
10 418


## Task 6: Smallest and largest

Use `sort` and `limit(1)`. Show only `Location` and the field you are asking about.

**6a.** Which local authority has the most females aged over 61?
**6b.** Which London borough has the fewest males under 18?
**6c.** Which local authority has the fewest males of all ages?

In [39]:
la = {"CODE": {"$regex": "^(E06|E07|E08|E09|W06|S12|N09)"}}

print("6a", list(female.find(la, {"_id": 0, "Location": 1, ">61": 1}).sort(">61", -1).limit(1)))
print("6b", list(male.find({"CODE": {"$regex": "^E09"}}, {"_id": 0, "Location": 1, "<18": 1}).sort("<18", 1).limit(1)))
print("6c", list(male.find(la, {"_id": 0, "Location": 1, "AllAges": 1}).sort("AllAges", 1).limit(1)))

6a [{'Location': 'Birmingham', '>61': 98449}]
6b [{'Location': 'City of London', '<18': 528}]
6c [{'Location': 'Isles of Scilly', 'AllAges': 1148}]


**Answers.** 6a Birmingham (98,449). 6b City of London (528). 6c Isles of Scilly (1,148).

## Task 7: Team challenge

Answer each question and **show the query** you used, not just the answer.

**7a.** How many Welsh areas have more than 20,000 females aged over 61? Name them.
**7b.** Find a Scottish council area (`S12`) whose name contains `Ayrshire`. How many are there?
**7c.** In Peterborough, are there more males or females aged 19–25?

In [40]:
query = {"CODE": {"$regex": "^W06"}, ">61": {"$gt": 20000}}
print("7a", female.count_documents(query))
for doc in female.find(query, {"_id": 0, "Location": 1, ">61": 1}).sort(">61", -1):
    print("   ", doc)

ayr = {"CODE": {"$regex": "^S12"}, "Location": {"$regex": "Ayrshire"}}
print("7b", male.count_documents(ayr), [d["Location"] for d in male.find(ayr)])

m = male.find_one({"Location": "Peterborough"})["19-25"]
f = female.find_one({"Location": "Peterborough"})["19-25"]
print("7c males:", m, "| females:", f)

7a 7
    {'Location': 'Cardiff', '>61': 34146}
    {'Location': 'Swansea', '>61': 31456}
    {'Location': 'Rhondda Cynon Taf', '>61': 29432}
    {'Location': 'Carmarthenshire', '>61': 27236}
    {'Location': 'Caerphilly', '>61': 22208}
    {'Location': 'Powys', '>61': 21788}
    {'Location': 'Flintshire', '>61': 20193}
7b 3 ['East Ayrshire', 'North Ayrshire', 'South Ayrshire']
7c males: 8419 | females: 7800


**Answers.** 7a 7 areas: Cardiff, Swansea, Rhondda Cynon Taf, Carmarthenshire, Caerphilly, Powys, Flintshire. 7b 3: East Ayrshire, North Ayrshire, South Ayrshire. 7c 8,419 males and 7,800 females: more males.